<a href="https://colab.research.google.com/github/05haileywang/ECON3916_PS3/blob/main/ECON3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install -q duckdb

import numpy as np
import pandas as pd
import duckdb

rng = np.random.default_rng(0)
con = duckdb.connect()

In [28]:
n_cust = 3000
n_west = 2000

n_orders = 1 + rng.negative_binomial(0.8, 0.12, n_cust)
revenue = np.maximum(50, 60 * n_orders + rng.normal(400, 150, n_cust))

customers = pd.DataFrame({
    "customer_id": np.arange(n_cust),
    "region": ["West"] * n_west + ["East"] * (n_cust - n_west),
    "revenue": revenue.round(2),
    "signup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 700, n_cust), unit="D")
})

orders = pd.DataFrame({
    "customer_id": np.repeat(customers["customer_id"].to_numpy(), n_orders)
})
orders["order_id"] = np.arange(len(orders))

con.register("customers", customers)
con.register("orders", orders)

print(con.sql("SELECT COUNT(*) AS customers FROM customers").df())
print(con.sql("SELECT COUNT(*) AS orders FROM orders").df())
print(con.sql("SELECT region, COUNT(*) AS customers FROM customers GROUP BY region").df())

   customers
0       3000
   orders
0   20615
  region  customers
0   East       1000
1   West       2000


Part 1.1: Customers contains 3000 customers, with 2,000 in the West region and 1000 in the East region. Orders contains 20,615 orders, with an average of 6.87 orders per customer.

In [29]:
query1 = """
SELECT
    AVG(revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers
WHERE region = 'West'
"""
con.execute(query1).df()

,avg_revenue,row_count
0,808.461105,2000


In [30]:
query2 = """
SELECT
    AVG(c.revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""
con.execute(query2).df()

,avg_revenue,row_count
0,1236.318753,13627


In [31]:
avg1 = con.execute(query1).df()["avg_revenue"][0]
avg2 = con.execute(query2).df()["avg_revenue"][0]

ratio = avg2 / avg1

print("Average without join:", round(avg1, 2))
print("Average with join:", round(avg2, 2))
print("Ratio:", round(ratio, 2))

Average without join: 808.46
Average with join: 1236.32
Ratio: 1.53


Step 1.2: Query1 calcualtes the average revenue from customer, where each customer appears once. Query2 joined the customer and order tables by the customer ID and created multiple rows for customer who placed more than one order. The second joined query gives customers with more than one order more weight when calculating the averages. The Joined query produces a higher average as a result.

The number correct number is the average from Query1 because we want to measure average revenue per customer, not average revenue per order. Each customer should only be counted once, regardless of how many orders they placed.

In [32]:
def fanout(con, query_before, query_after):

    before = con.execute(query_before).df()["row_count"][0]
    after = con.execute(query_after).df()["row_count"][0]

    if before != after:
        print("Row count changed")
        print("Rows before:", before)
        print("Rows after:", after)
    else:
        print("Row count did not change")

# Count West customers before the join
before_query = """
SELECT COUNT(*) AS row_count
FROM customers
WHERE region = 'West'
"""

# Count West customers after the join
after_query = """
SELECT COUNT(*) AS row_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

# Test 1: with join
print("Test 1: Query with join")
fanout(con, before_query, after_query)

# Test 2: without join
print("\nTest 2: Query without join")
fanout(con, before_query, before_query)

Test 1: Query with join
Row count changed
Rows before: 2000
Rows after: 13627

Test 2: Query without join
Row count did not change


In [33]:
# Set random seed
rng = np.random.default_rng(2000)

# Randomly select 15% of customers to set to NULL
missing = rng.random(len(customers)) < 0.15
customers.loc[missing, "region"] = None

# Update the customers table in DuckDB
con.register("customers", customers)

# Count missing region values
query = """
SELECT
    COUNT(*) AS total_customers,
    COUNT(region) AS known_regions,
    COUNT(*) - COUNT(region) AS missing_regions
FROM customers
"""

con.execute(query).df()

,total_customers,known_regions,missing_regions
0,3000,2566,434


Part 2.1: I used a random number generator to select around 15% of the customers to replace their region with NULL to similate missing data points. I used SQL to count the total number of customers, the number with regions, and the number with missing regions.

In [34]:
# Query 1: Exclude West customers
query1 = """
SELECT COUNT(*) AS row_count
FROM customers
WHERE region != 'West'
"""

# Query 2: Include customers with missing regions
query2 = """
SELECT COUNT(*) AS row_count
FROM customers
WHERE region != 'West' OR region IS NULL
"""

# Run both queries
count1 = con.execute(query1).df()["row_count"][0]
count2 = con.execute(query2).df()["row_count"][0]

# Compare results
print("Original query:", count1)
print("Corrected query:", count2)
print("Difference:", count2 - count1)

Original query: 867
Corrected query: 1301
Difference: 434


Part 2.2:The original query excluded 434 customers because NULL returns UNKNOWN, and WHERE clause retains rows where the condition is determined to be TRUE.

If the customers with missing regions are systematically different from customers with regions, then excluding them coud introduce selection bias into our model.

As a result, regional comparisons may not accurately represent the company's customer base. The direction and size of the bias would depend on which customers are missing and how they differ from those included in the analysis.

In [35]:
# Calculate average revenue for each region using a window function
query = """
SELECT
    customer_id,
    region,
    revenue,
    AVG(revenue) OVER (PARTITION BY region) AS region_avg
FROM customers
"""

# Run the query
result = con.execute(query).df()

# Display the first 10 rows
print(result.head(10))

# Check that the row count did not change
original_count = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]

print("\nOriginal row count:", original_count)
print("Window function row count:", len(result))

   customer_id region  revenue  region_avg
0         1005   West  1048.57  809.270394
1         1498   West  1217.16  809.270394
2          503   West  1985.94  809.270394
3          505   West  1416.91  809.270394
4            5   West   383.38  809.270394
5            6   West   923.92  809.270394
6            7   West   386.01  809.270394
7            8   West   208.18  809.270394
8           11   West   227.53  809.270394
9           12   West   273.22  809.270394

Original row count: 3000
Window function row count: 3000


Step 2.3: A windom function was used to calculate the average revenue for each region, but it also kept each customer's individual revenue. The PARTITION BY was used to group the customers by their region to calculate their average without combining the rows.  

The orignal customers table and the windom function both contain 3000 rows. This means that the window function did not change the number of observations, unlike the JOIN in part 1.

In [36]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Set random seed
rng = np.random.default_rng(2000)

# Create 5000 customers
n = 5000

# Simulate existing account size
account_size = rng.normal(100, 30, n)

# Larger accounts are more likely to be in the West
prob_west = np.where(account_size > 100, 0.8, 0.2)

# Assign customers to regions
west = rng.binomial(1, prob_west)

# Set the true effect of the programme
true_effect = 30

# Simulate revenue
revenue = 500 + 8 * account_size + true_effect * west + rng.normal(0, 100, n)

# Create dataframe
program = pd.DataFrame({
    "account_size": account_size,
    "west": west,
    "revenue": revenue
})

# Register table in DuckDB
con.register("program", program)

# Calculate average revenue for each group
naive_query = """
SELECT
    west,
    AVG(revenue) AS avg_revenue
FROM program
GROUP BY west
"""

print("Average revenue by programme:")
print(con.execute(naive_query).df())

# Calculate naive difference in average revenue
naive = con.execute("""
SELECT
    AVG(CASE WHEN west = 1 THEN revenue END) -
    AVG(CASE WHEN west = 0 THEN revenue END)
FROM program
""").fetchone()[0]

# Adjust for existing account size using regression
model = smf.ols("revenue ~ west + account_size", data=program).fit()

adjusted = model.params["west"]

# Compare estimates
print("\nTrue programme effect:", true_effect)
print("Naive estimate:", round(naive, 2))
print("Adjusted estimate:", round(adjusted, 2))

Average revenue by programme:
   west  avg_revenue
0     0  1184.839421
1     1  1447.922930

True programme effect: 30
Naive estimate: 263.08
Adjusted estimate: 30.84


The true effect of the account management program was set to 30.00. The naive compasrison estimated an effect of 263.08 which is 233.08 greater than the true effect. This is likely due to the fact that customers with larger accounts were more likely to be assigned to the West. The larger the account, the more revenue it generates, so the naive comparison overestimates the program's effectiveness.

After controlling for existing account size, the adjusted estimate decreased to 30.84, which is slightly greater than the true effect. This demonstrates the power of confounding variables and their ability to influence estimates of a program's effectiveness.

In [37]:
# Set random seed
rng = np.random.default_rng(2000)

# Simulate support tickets
program["support_tickets"] = (
    5 * program["west"]
    + 0.02 * program["revenue"]
    + rng.normal(0, 2, len(program))
)

# Register updated table in DuckDB
con.register("program", program)

# Regression without support tickets
model1 = smf.ols("revenue ~ west + account_size", data=program).fit()

# Regression with support tickets
model2 = smf.ols("revenue ~ west + account_size + support_tickets", data=program).fit()

# Compare estimates
adjusted = model1.params["west"]
with_tickets = model2.params["west"]

print("True programme effect:", true_effect)
print("Adjusted estimate without tickets:", round(adjusted, 2))
print("Adjusted estimate with tickets:", round(with_tickets, 2))

print("\nBias without tickets:", round(adjusted - true_effect, 2))
print("Bias with tickets:", round(with_tickets - true_effect, 2))

True programme effect: 30
Adjusted estimate without tickets: 30.84
Adjusted estimate with tickets: -250.0

Bias without tickets: 0.84
Bias with tickets: -280.0


Initially, support tickets seem like a useful measure of customer engagement, but when we controlled for them, it resulted in the programme's effectiveness estimate becoming less accurate. Customers who are involved in the customer programme will likely submit more tickets, and customers who have higher revenue will also likely submit more tickets because they need more support. Introducing support tickets into the regression can introduce bias. Comparing customers with the same ticket count is misleading becuase the program's participation increases support tickets, so althought two customers might have a similar ticket count, their support needs may be different. Our original adjusted estimate was 30.84, which was close to the true programme effect of 30.00. However, after controlling for support tickets, the estimated effect decreased to -250.00, which was 280.00 below the true effect. This demonstrates how adding support tickets made our estimate significantly less accurate. Therefore, I would recommend controlling for account size and excluding support tickets when running the regression.

Model: GPT-6
Date: October 29, 2026

[Prep] Act as an analytics engineer who reviews SQL before it reaches a dashboard.
[Request] Write a Python function review_query(con, sql) that runs against DuckDB and returns a list of warnings for a given SELECT. It should flag at minimum: a join that changes the row count, a filter on a nullable column that will silently drop unknowns, and an aggregate computed over a fanned-out result.
[Iterate] duckdb and pandas only. Reuse my table names, customers and orders. Return structured warnings, not printed text, so I can test it.
[Mechanism Check] Add comments explaining how you detect fan-out without running the full join twice, and why three-valued logic makes the null filter a correctness issue rather than a style issue.
[Evaluate] Run it against both of my Phase 1 queries and tell me whether it would have caught the $1,240 before it reached the board.

1. Python Function: review_query(con, sql)

The goal is to catch SQL queries that execute successfully but produce misleading results.

The Phase 1 example illustrates this problem:

- Correct query: Calculates average revenue using one row per customer, approximately 808
- Incorrect query: Joins customers to orders before calculating average customer revenue, producing approximately 1,236
- Problem: Customers with more orders appear multiple times, giving them more weight in the average

The function checks for three issues:

1. fan_out: A join increases the number of rows relative to the customer population being analyzed
2. nullable_filter: A WHERE condition may exclude rows with NULL values
3. aggregate_after_fan_out: An aggregate such as AVG(c.revenue) is calculated after a join duplicates customer records

Each warning is returned as a dictionary containing a type, severity, and message, making it possible to test the results in Colab.


2. Python Code

The function uses only duckdb, pandas, and Python's built-in re module.

It is designed for the customers and orders queries in the assignment rather than attempting to parse every possible SQL statement.

import duckdb
import pandas as pd
import re

def review_query(con, sql):
    warnings = []
    query = " ".join(sql.strip().split())
    q = query.lower()

    if not q.startswith(("select ", "with ")):
        raise ValueError("Only SELECT queries are supported")

    # Validate the query without fetching its result
    con.execute("EXPLAIN " + sql)

    def add_warning(type, severity, message):
        warnings.append({
            "type": type,
            "severity": severity,
            "message": message
        })

    # Identify whether customers is joined to orders
    join_match = re.search(
        r"\b(left\s+join|inner\s+join|join)\s+orders\b",
        q
    )

    has_join = join_match is not None

    # Check filters on nullable customer columns
    columns = con.execute("DESCRIBE customers").df()

    nullable_columns = columns.loc[
        columns["null"] == "YES", "column_name"
    ].str.lower().tolist()

    where_match = re.search(
        r"\bwhere\b(.*?)(?:\bgroup\s+by\b|\border\s+by\b|\bhaving\b|$)",
        q
    )

    where_clause = where_match.group(1) if where_match else ""

    for col in nullable_columns:
        pattern = rf"\b(?:c\.|customers\.)?{re.escape(col)}\b"

        if re.search(pattern, where_clause):
            # SQL uses three-valued logic: TRUE, FALSE, UNKNOWN
            # Comparisons involving NULL usually produce UNKNOWN
            # WHERE retains only TRUE, so NULL rows disappear
            # This changes the population being analyzed
            add_warning(
                "nullable_filter",
                "medium",
                f"Filtering on nullable column '{col}' may silently exclude NULL rows"
            )

    if has_join:
        # Detect fan-out using order counts per customer
        # This avoids executing the full customers-orders join twice
        # Each customer with multiple matching orders creates multiple rows

        order_counts = con.execute("""
            SELECT customer_id, COUNT(*) AS n_orders
            FROM orders
            GROUP BY customer_id
        """).df()

        # A customer with more than one order creates fan-out
        repeated_customers = (
            order_counts["n_orders"] > 1
        ).sum()

        if repeated_customers > 0:
            add_warning(
                "fan_out",
                "high",
                f"{repeated_customers} customers have multiple orders, "
                "so the join can increase the row count"
            )

            # Check whether customer-level aggregates follow the join
            aggregate_pattern = (
                r"\b(avg|sum|count)\s*\(\s*"
                r"(?:c\.|customers\.)?(revenue|customer_id)\b"
            )

            if re.search(aggregate_pattern, q):
                add_warning(
                    "aggregate_after_fan_out",
                    "high",
                    "Customer-level aggregate is calculated after "
                    "joining orders. Customers with more orders "
                    "receive greater weight in the result"
                )

    return warnings


3. How the Function Detects Fan-Out

The important section is:

order_counts = con.execute("""
    SELECT customer_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY customer_id
""").df()

repeated_customers = (
    order_counts["n_orders"] > 1
).sum()

Instead of repeatedly executing the full join, the function groups orders by customer_id.

If a customer has five orders, joining that customer to orders produces five rows instead of one.

This is sufficient to identify potential fan-out in a one-to-many relationship.

However, the function does not prove that every flagged query has an increased row count. It identifies potential fan-out based on order multiplicity.

It also does not fully parse complex SQL involving nested queries, aliases, or multiple joins.


4. Why NULL Filtering Is a Correctness Issue

SQL uses three-valued logic:

- TRUE
- FALSE
- UNKNOWN

When a nullable column is compared with a value, a NULL generally produces UNKNOWN.

For example:

SELECT *
FROM customers
WHERE region = 'West';

Customers whose region is NULL are excluded because WHERE retains only rows where the condition evaluates to TRUE.

This is not merely a formatting or coding-style issue. It changes which customers are included in the analysis and can introduce selection bias.

The warning identifies that risk. Whether excluding unknown regions is actually incorrect depends on the intended population.


5. Evaluate Against Phase 1 Query 1

Query 1 calculates average revenue without a join.

query1 = """
SELECT
    AVG(revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers
WHERE region = 'West'
"""

warnings1 = review_query(con, query1)

print(warnings1)

Expected output:

[]

The query calculates average revenue directly from the customers table.

There is no join, so customers are not duplicated.

The expected average is approximately $812.

The function would flag a nullable filter only if the region column is declared nullable in the table schema.


6. Evaluate Against Phase 1 Query 2

Query 2 calculates average revenue after joining orders.

query2 = """
SELECT
    AVG(c.revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

warnings2 = review_query(con, query2)

print(warnings2)

Expected warning types:

[
    "fan_out",
    "aggregate_after_fan_out"
]

The function detects that customers can have multiple orders, meaning the join duplicates customer-level observations.

It then identifies that AVG(c.revenue) is calculated after the join.

The resulting average is approximately $1,240 rather than the correct customer-level average of approximately $812.

The function may also produce a nullable_filter warning if region is declared nullable.


7. Test Whether the Function Catches the Error

The following assertions can be used in Colab to verify the behavior automatically.

# Review both queries
warnings1 = review_query(con, query1)
warnings2 = review_query(con, query2)

# Extract warning types
types1 = [w["type"] for w in warnings1]
types2 = [w["type"] for w in warnings2]

# Check expected behavior
assert "fan_out" not in types1
assert "aggregate_after_fan_out" not in types1

assert "fan_out" in types2
assert "aggregate_after_fan_out" in types2

print("All tests passed!")

These tests verify that the function identifies the expected warning types.

They do not independently verify the revenue calculations.


8. Evaluation

Query 1:
- Join fan-out: No warning
- Nullable filter: Depends on schema
- Aggregate after fan-out: No warning
- Revenue estimate: Approximately $812
- Should be flagged for review: No fan-out issue

Query 2:
- Join fan-out: Warning
- Nullable filter: Depends on schema
- Aggregate after fan-out: Warning
- Revenue estimate: Approximately $1,240
- Should be flagged for review: Yes


9. Would the Function Have Caught the $1,240 Before It Reached the Board?

Yes, assuming the tables have the expected one-to-many relationship and the reviewer is run before dashboard publication.

The second query would trigger high-severity warnings for fan-out and aggregation after fan-out.

However, the function does not automatically correct the result or block publication. It identifies the structural problem so an analyst can investigate before approving the query.

The $1,240 error could have been detected before publication because the query calculates a customer-level average after changing the unit of observation from customers to orders.

Customers with more orders are represented multiple times, giving them greater weight in the calculation. This produces a misleading estimate of average customer revenue.

The function would identify this issue and warn the analyst before the query is used in a dashboard.


10. Limitations

The function is a simplified SQL reviewer designed for the customers and orders tables.

It does not fully support complex SQL queries, including nested queries, multiple joins, and unusual alias structures.

The fan-out detection identifies customers with multiple orders but does not always establish whether the actual query changes row counts.

The nullable-filter detection relies on the database schema. A column may contain NULL values even when its schema does not explicitly identify it as nullable.

The function also does not automatically prevent a problematic query from being published.

Finally, the Phase 1 SQL statements shown above are illustrative reconstructions rather than verified copies of the original queries. The actual DuckDB tables were not available for execution, so the reported warning results are expected outcomes rather than confirmed test results.

In [38]:
import duckdb
import pandas as pd
import re

def review_query(con, sql):
    warnings = []
    query = " ".join(sql.strip().split())
    q = query.lower()

    if not q.startswith(("select ", "with ")):
        raise ValueError("Only SELECT queries are supported")

    # Validate the query without fetching its result
    con.execute("EXPLAIN " + sql)

    def add_warning(type, severity, message):
        warnings.append({
            "type": type,
            "severity": severity,
            "message": message
        })

    # Identify whether customers is joined to orders
    join_match = re.search(
        r"\b(left\s+join|inner\s+join|join)\s+orders\b",
        q
    )

    has_join = join_match is not None

    # Check filters on nullable customer columns
    columns = con.execute("DESCRIBE customers").df()

    nullable_columns = columns.loc[
        columns["null"] == "YES", "column_name"
    ].str.lower().tolist()

    where_match = re.search(
        r"\bwhere\b(.*?)(?:\bgroup\s+by\b|\border\s+by\b|\bhaving\b|$)",
        q
    )

    where_clause = where_match.group(1) if where_match else ""

    for col in nullable_columns:
        pattern = rf"\b(?:c\.|customers\.)?{re.escape(col)}\b"

        if re.search(pattern, where_clause):
            # SQL uses three-valued logic: TRUE, FALSE, UNKNOWN
            # Comparisons involving NULL usually produce UNKNOWN
            # WHERE retains only TRUE, so NULL rows disappear
            # This changes the population being analyzed
            add_warning(
                "nullable_filter",
                "medium",
                f"Filtering on nullable column '{col}' may silently exclude NULL rows"
            )

    if has_join:
        # Detect fan-out using order counts per customer
        # This avoids executing the full customers-orders join twice
        # Each customer with multiple matching orders creates multiple rows

        order_counts = con.execute("""
            SELECT customer_id, COUNT(*) AS n_orders
            FROM orders
            GROUP BY customer_id
        """).df()

        # A customer with more than one order creates fan-out
        repeated_customers = (
            order_counts["n_orders"] > 1
        ).sum()

        if repeated_customers > 0:
            add_warning(
                "fan_out",
                "high",
                f"{repeated_customers} customers have multiple orders, "
                "so the join can increase the row count"
            )

            # Check whether customer-level aggregates follow the join
            aggregate_pattern = (
                r"\b(avg|sum|count)\s*\(\s*"
                r"(?:c\.|customers\.)?(revenue|customer_id)\b"
            )

            if re.search(aggregate_pattern, q):
                add_warning(
                    "aggregate_after_fan_out",
                    "high",
                    "Customer-level aggregate is calculated after "
                    "joining orders. Customers with more orders "
                    "receive greater weight in the result"
                )

    return warnings

In [39]:
# Review Phase 2 queries
warnings1 = review_query(con, query1)
warnings2 = review_query(con, query2)

print("Query 1:")
print(warnings1)

print("\nQuery 2:")
print(warnings2)

Query 1:
[{'type': 'nullable_filter', 'severity': 'medium', 'message': "Filtering on nullable column 'region' may silently exclude NULL rows"}]

Query 2:
[{'type': 'nullable_filter', 'severity': 'medium', 'message': "Filtering on nullable column 'region' may silently exclude NULL rows"}]


In [40]:
import duckdb
import pandas as pd
import re

def review_query(con, sql):
    warnings = []
    query = " ".join(sql.strip().split())
    q = query.lower()

    if not q.startswith(("select ", "with ")):
        raise ValueError("Only SELECT queries are supported")

    # Validate the query without fetching its result
    con.execute("EXPLAIN " + sql)

    def add_warning(type, severity, message):
        warnings.append({
            "type": type,
            "severity": severity,
            "message": message
        })

    # Identify whether customers is joined to orders
    join_match = re.search(
        r"\b(left\s+join|inner\s+join|join)\s+orders\b",
        q
    )

    has_join = join_match is not None

    # Check filters on nullable customer columns
    columns = con.execute("DESCRIBE customers").df()

    nullable_columns = columns.loc[
        columns["null"] == "YES", "column_name"
    ].str.lower().tolist()

    where_match = re.search(
        r"\bwhere\b(.*?)(?:\bgroup\s+by\b|\border\s+by\b|\bhaving\b|$)",
        q
    )

    where_clause = where_match.group(1) if where_match else ""

    for col in nullable_columns:
        pattern = rf"\b(?:c\.|customers\.)?{re.escape(col)}\b"

        # Check whether the query explicitly includes NULL values
        null_check = rf"\b(?:c\.|customers\.)?{re.escape(col)}\s+is\s+null\b"

        if re.search(pattern, where_clause) and not re.search(null_check, where_clause):

            # SQL uses three-valued logic: TRUE, FALSE, UNKNOWN
            # Comparisons involving NULL produce UNKNOWN
            # WHERE only retains TRUE, so NULL rows disappear
            add_warning(
                "nullable_filter",
                "medium",
                f"Filtering on nullable column '{col}' may silently exclude NULL rows"
            )

    if has_join:
        # Count orders per customer instead of executing the full join twice
        # Customers with multiple orders can create duplicate customer rows

        order_counts = con.execute("""
            SELECT customer_id, COUNT(*) AS n_orders
            FROM orders
            GROUP BY customer_id
        """).df()

        repeated_customers = (
            order_counts["n_orders"] > 1
        ).sum()

        if repeated_customers > 0:
            add_warning(
                "fan_out",
                "high",
                f"{repeated_customers} customers have multiple orders, "
                "so the join can increase the row count"
            )

            # Check whether customer-level aggregates follow the join
            aggregate_pattern = (
                r"\b(avg|sum|count)\s*\(\s*"
                r"(?:c\.|customers\.)?(revenue|customer_id)\b"
            )

            if re.search(aggregate_pattern, q):
                add_warning(
                    "aggregate_after_fan_out",
                    "high",
                    "Customer-level aggregate is calculated after "
                    "joining orders. Customers with more orders "
                    "receive greater weight in the result"
                )

    return warnings

In [41]:
# Test the AI reviewer with my changes
print("Query 1:")
print(review_query(con, query1))

print("\nQuery 2:")
print(review_query(con, query2))

Query 1:
[{'type': 'nullable_filter', 'severity': 'medium', 'message': "Filtering on nullable column 'region' may silently exclude NULL rows"}]

Query 2:
[]


Part 4.1: The AI generated code correctly identifies in Query1 that the filter may exclude NULL rows. However, in Query2, the filter design was changed. Query2 included all missing regions OR region IS NULL. The AI generated code falsely flagged this error. To correct this issue, I would change the function to also check if the SQL query has an additional condition that affects the NULL values. This would provide necessary nuance to differentiate the two queries and prevent a false positive flag.

I put in a small modificiation to check whether the query contains a condition IS NULL before outputting the warning. After I made this change to the code, Query1 returned the same warning, but Query2 output an empty list.

In [43]:
# Test the Phase 1 revenue queries

print("Query without join")
print(review_query(con, """
SELECT
    AVG(revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers
WHERE region = 'West'
"""))

print("\nQuery with join")
print(review_query(con, """
SELECT
    AVG(c.revenue) AS avg_revenue,
    COUNT(*) AS row_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""))

Query without join
[{'type': 'nullable_filter', 'severity': 'medium', 'message': "Filtering on nullable column 'region' may silently exclude NULL rows"}]

Query with join
[{'type': 'nullable_filter', 'severity': 'medium', 'message': "Filtering on nullable column 'region' may silently exclude NULL rows"}, {'type': 'fan_out', 'severity': 'high', 'message': '2428 customers have multiple orders, so the join can increase the row count'}, {'type': 'aggregate_after_fan_out', 'severity': 'high', 'message': 'Customer-level aggregate is calculated after joining orders. Customers with more orders receive greater weight in the result'}]


Part 4.1: I also test the improved AI code against the Phase 1 queries. For the query that did not have a join, it was able to indentify the null regions. This flag is correct, but it misses the nuance that some of the unknown regions were intentional and a part of calculating average revenue for West. For the other query that did not have a join, it identified that there is null regions, fan out, and agreegation after fan out. It correcetly was able to identify that 2,428 customers had more order and were given more weight in the average revenue calculation. It would have been able to identify the issues before the outcome of the inflated figures.